In [ ]:
"""
Before we can do vector search, we need to turn our text into vectors. 
We call this process embedding: we embed text into a vector space. The vectors we get back are also called "embeddings."
"""

In [1]:
"""
Choosing the model:

We'll use all-MiniLM-L6-v2:
* 384-dimensional vectors (compact)
* Fast on CPU
* Good quality for general English text
* Use cosine for similarity check
"""

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

/mnt/c/Users/xie_j/Documents/llm-zoomcamp-code/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12090). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
"""
The first time running this, it downloads the model (~80 MB) and the tokenizer from HuggingFace. 
The tokenizer turns text into something the model can read. After that, both load from a local cache.
"""

# trying with simple examples:

q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [3]:
"""
v1 is a vector, an array of 384 numbers. Each number stands for some concept the model learned. We can't read off what any one of them means. 
But two vectors with similar values point to texts about similar things.
"""

# now encoding our document:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [4]:
# Next, we compare the query against the document using dot product:

v1.dot(dv)

np.float32(0.323324)

In [5]:
# trying an unrelated query:

q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

v2.dot(dv)

np.float32(0.019730514)

In [ ]:
"""
In practice, we rarely get cosine similarity below 0. 
The embedding model maps text to a region of the vector space where most vectors have positive components. 
There's no concept of "opposite meaning" that maps to a vector pointing the other way.

"""

In [6]:
# Using the ingest.py from module 1

! wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-28 10:48:07--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 888 [text/plain]
Saving to: ‘ingest.py.3’

ingest.py.3         100%[===================>]     888  --.-KB/s    in 0s      

2026-06-28 10:48:07 (29.9 MB/s) - ‘ingest.py.3’ saved [888/888]



In [7]:
# Loading the data

from ingest import load_faq_data

documents = load_faq_data()

In [8]:
"""
Generating embeddings:

Build one text per document (Each document is a Python dictionary with a question and an answer. We embed both together.
That way, a query can match against the question text and the answer text in our index.
"""

texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [9]:
"""
Now we generate the embeddings. We have about 1200 texts here. We won't hand the model all of them at once. 
That takes a long time, and we can't see what's happening inside. Instead we split them into batches.
"""

from tqdm.auto import tqdm     # to watch the progress

# chunck the dataset into batches of 50 and encode each batch:

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors) # adding one by one from batch_vectors

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [12]:
print(batch_vectors[0])

[ 4.54983115e-02  8.08303803e-02 -1.26925623e-02 -9.27333683e-02
  5.39263263e-02 -8.95349905e-02  3.58852744e-02 -1.53941670e-02
  9.01577771e-02  3.18809636e-02 -4.49900664e-02 -1.27348572e-01
  1.93041458e-03 -6.40360964e-03  5.82296513e-02 -8.89594480e-02
 -3.90619934e-02 -2.96584517e-02  9.05089546e-04  1.25079718e-03
 -7.99255744e-02  4.53624353e-02 -5.65566774e-03  1.43255182e-02
 -5.44453971e-02 -5.20810559e-02 -3.64351459e-02  1.52322408e-02
 -5.47269061e-02 -1.89798530e-02  1.11576982e-01  3.95277999e-02
  1.43629480e-02 -1.84233747e-02  2.05298848e-02  2.89582200e-02
 -1.84498169e-02 -3.14882472e-02  1.06745213e-02 -1.84888877e-02
  9.78776152e-05  1.51512548e-02  2.52970285e-03 -7.67144635e-02
 -3.48150148e-03 -2.14430746e-02 -8.16590525e-03 -9.90954507e-03
  1.37655251e-02  2.04346068e-02 -4.10980452e-03 -2.93786693e-02
  6.31127954e-02 -5.96949831e-02 -3.94634157e-02  1.10618204e-01
 -2.13480294e-02  7.47054443e-02  3.34601216e-02 -1.23764820e-01
 -7.18071386e-02 -6.43496

In [10]:
"""
We turn them into a 2-dimensional array (matrix) where
* rows are documents (vectors)
* columns are dimensions of the vectors
"""

import numpy as np
X = np.array(vectors)

X.shape # returns number of documents vs number of dimensions

(1350, 384)

In [15]:
X[0,0]

np.float32(-0.026706176)

In [11]:
"""
Vector Search:

We have a matrix X with all document embeddings. 
We take a query, compare it against every document, and keep the most similar ones.
"""

# embedding a query
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

# computing the dot product against all documents:
# This is matrix-vector multiplication. 
# Each element i of scores is the cosine similarity between document i (row i of X) and v_query.
scores = X.dot(v_query)

In [ ]:
# alternatively:
scores = [v_query.dot(X[i]) for i in range(len(X))]

"""
But X.dot(v_query) is much faster. Numpy runs optimized C code instead of looping in Python, 
so the matrix version is hard to beat. The outcome is the same either way: one score per document.
"""

In [12]:
"""
Best match: the highest score is the most similar document
"""

idx = np.argmax(scores) # get the index of the highest score
idx, scores[idx]

(np.int64(2), np.float32(0.762941))

In [18]:
# checking what document this index corresponds to:

documents[idx]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [19]:
"""
Top 5 results: Usually we want more than the single best match, so let's pull the top 5.
"""

# np.argsort sorts from lowest to highest, so the last 5 are the top ones:
top5 = np.argsort(scores)[-5:]

# They come out smallest-first, so we reverse them to get the highest first:
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [21]:
# reading the top 5
scores[top5]

array([0.762941  , 0.7579373 , 0.7192131 , 0.6536312 , 0.56009996],
      dtype=float32)

In [22]:
# reading the docs that correspond to the top 5:

for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()
    

0.762941
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579373
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192131
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 

In [ ]:
"""
We return 5 and not the single best for a reason. The answer to a question can be spread across several documents. 
One holds part of it, another fills in the rest. Sometimes the top result isn't the right one but the second is. 
We send all 5 to the LLM and let it combine them.
"""

In [ ]:
"""
Until now, we did vector search by hand with numpy. We embedded the query, computed dot products, and found the best matches. 
Writing the argsort and matrix code every time gets old, and it can't filter by course. So instead we'll use a library that wraps all of it.
We will use minsearch
"""

In [13]:
from minsearch import VectorSearch

# indexing the documents, scores
vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [15]:
# searching

query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

# Under the hood it does the same thing we just did by hand. It computes the dot product between each vector (after filtering) and our query vector.
results[0]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

In [17]:
# filtering by course:

results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [19]:
"""
RAG with Vector Search
"""

# from module 1:

def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

"""
The search step used keyword search. Now we swap in vector search. 
Because RAG is modular, search is the only step we touch. Build prompt and the LLM call stay exactly as they were.
"""


'\nThe search step used keyword search. Now we swap in vector search. \nBecause RAG is modular, search is the only step we touch. Build prompt and the LLM call stay exactly as they were.\n'

In [20]:
! wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
! wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-28 10:56:52--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.2’

rag_helper.py.2     100%[===================>]   2.08K  --.-KB/s    in 0.002s  

2026-06-28 10:56:52 (1.12 MB/s) - ‘rag_helper.py.2’ saved [2134/2134]

--2026-06-28 10:56:52--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 20

In [21]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [22]:
# downloading the index and data:

from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [24]:
# using the RAGBase class:

from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.'

In [26]:
"""
Replacing search by vector search:

We can't pass vindex to RAG as-is. Text search takes the query string directly,
but vector search needs the query as a vector first. 
So we subclass RAGBase and override search to encode the query before searching.
"""


class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )


"""
The __init__ method adds one extra argument, embedder, for the sentence transformer. 
Inside search we use it to turn the query into a vector. 
Then we query vindex with that vector instead of the raw text. Everything else is inherited from RAGBase.
"""

'\nThe __init__ method adds one extra argument, embedder, for the sentence transformer. \nInside search we use it to turn the query into a vector. \nThen we query vindex with that vector instead of the raw text. Everything else is inherited from RAGBase.\n'

In [27]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [28]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join. You can start learning and submitting homework while the form is open. If you want a certificate, make sure to submit your project before submissions close.'

In [ ]:
"""
NN - Nearest Neighbor - (exact):    compare query against ALL documents -> top 5
ANN - Approximate  NN - (approx):  narrow down to a region -> compare within region -> top 5
"""

In [31]:
# sqlitesearch is the persistent sibling of minsearch, and it solves both problems at once.

"""
sqlitesearch supports three ANN modes:
* lsh (default): up to 100K vectors, random hyperplane projections
* ivf: 10K-500K vectors, K-means clustering
* hnsw: 10K-1M+ vectors, proximity graph (highest recall)
"""

# For our small dataset, lsh is fine. All modes use two-phase search: approximate candidate retrieval, then exact cosine similarity reranking.

from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [33]:
"""
indexing the data:
The index is saved to faq_vectors2.db. Unlike minsearch, this file persists on disk. 
You can search immediately after indexing, or reopen the index later without re-indexing.
"""

vs_index.fit(vectors, documents)

ValueError: Index already contains documents. Use clear() to reset the index or add() to append documents.

In [35]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
  'doc_id': '41aabbd7c5'},
 {'course': 'mlops-zoomcamp',
  'section': 'General Cour

In [36]:
# filtering by course:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [37]:
vs_index.close()